In [45]:
import csv
import json
from genraweb.resources import DB
from deepdiff import DeepDiff

In [46]:
rows = []
with open('toxref_mapping.csv', 'r') as csvfile:
    for row in csv.reader(csvfile):
        rows.append(row)
rows = rows[1:]  # header

def ret_ds_form(provided):
    first, second = provided.split('_', 1)
    second = second.replace("_", " ")
    return first.upper() + ":" + second

# we only care about the ones with notes, fillter out empty ones
fp_stats = {ret_ds_form(row[0]): row[2] for row in rows if row[2]}
fp_stats_db = { doc["ds"]: doc["notes"] for doc in DB.fp_stats.find({}) if doc.get("ds") and doc.get("notes")}
diff = DeepDiff(fp_stats, fp_stats_db)

# this shows fp_stats on mongo DB is up to date, let's save that one instead of the CSV because it's got more data
print(diff)

{'values_changed': {"root['ACU:eye']": {'new_value': 'Effects could include opacity.', 'old_value': 'Effects could include opacity. '}}}


In [52]:
# write to local
with open('misc/toxref_notes.json', 'w') as f:
    json.dump(list(DB.fp_stats.find({}, {"_id": False})), f)